# arXiv Paper Scraper — Exploratory Analysis

Use this notebook to interactively explore your scraped data.

In [14]:
import json
import pandas as pd
import numpy as np
import plotly.express as px
from collections import Counter

## 1. Load scraped data

In [15]:
path = "../data/raw/news_full.json"

with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

print("Shape:", df.shape)
df.head()

Shape: (146, 10)


,title,link,source,content,sentiment_label,sentiment_score,entities,embedding,cluster,clean_text
0,Hegseth says clock paused on deadline to seek ...,https://www.bbc.com/news/articles/cvgz7l5v03po,bbc,US Defence Secretary Pete Hegseth has argued t...,NEGATIVE,0.991885,"[[US, GPE], [Pete Hegseth, PERSON], [Trump, PE...","[0.1753679436, 0.1463331636, 0.1060355869, 0.0...",4,US Defence Secretary Pete Hegseth has argued t...
1,Trump says he will hike tariffs on EU cars to ...,https://www.bbc.com/news/articles/c4g8zpylzz9o,bbc,Donald Trump is increasing the tariffs charged...,POSITIVE,0.993924,"[[Donald Trump, PERSON], [the European Union, ...","[0.0277977228, 0.0568626814, 0.1383040599, -0....",1,Donald Trump is increasing the tariffs charged...
2,Watch: Moment WW2 bomb detonated in PlymouthA ...,https://www.bbc.com/news/videos/cn0pd1dxgydo,bbc,This is the moment a huge explosion was heard ...,POSITIVE,0.984098,"[[Plymouth, GPE], [250kg WW2, QUANTITY], [More...","[0.0771399288, 0.0919012434, 0.0290624432, 0.0...",4,This is the moment a huge explosion was heard ...
3,Missing Oscar found after Academy Award winner...,https://www.bbc.com/news/articles/cz72j59znw3o,bbc,An Oscar that went missing after its winner wa...,NEGATIVE,0.998204,"[[Oscar, PERSON], [New York, GPE], [BBC, ORG],...","[0.0976981592, 0.2068713849, -0.1244160768, 0....",4,An Oscar that went missing after its winner wa...
4,"War criminal Mladic close to death, say lawyer...",https://www.bbc.com/news/articles/c3e283g80n4o,bbc,Lawyers acting for convicted Bosnian Serb war ...,NEGATIVE,0.550064,"[[Bosnian Serb, NORP], [Ratko Mladic, LOC], [U...","[-0.0806638374, 0.1448248165, -0.3206217313, -...",3,Lawyers acting for convicted Bosnian Serb war ...


## 2. Distribuição de Sentimento

In [16]:
fig = px.histogram(
    df,
    x="sentiment_label",
    title="🧠 Sentiment Distribution",
    color="sentiment_label"
)
fig.show()

## 3. Distribuição de Clusters

In [27]:
fig = px.histogram(
    df,
    x="cluster",
    title="🧩 Topic Clusters",
)
fig.show()

## 4. Top Keywords (rápido)

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

texts = df["clean_text"].fillna("").tolist()

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=1000
)

X = vectorizer.fit_transform(texts)

scores = np.asarray(X.mean(axis=0)).ravel()
terms = vectorizer.get_feature_names_out()

top_idx = scores.argsort()[-20:]

keywords = [(terms[i], scores[i]) for i in top_idx]
keywords = sorted(keywords, key=lambda x: x[1])

kw_df = pd.DataFrame(keywords, columns=["keyword", "score"])

fig = px.bar(
    kw_df,
    x="score",
    y="keyword",
    orientation="h",
    title="🔑 Top Keywords"
)
fig.show()

## 5. Entidades mais comuns (NER)

In [19]:
all_entities = []

for ents in df["entities"].dropna():
    for text, label in ents:
        all_entities.append((text, label))

entity_counts = Counter(all_entities)

top_entities = entity_counts.most_common(20)

ent_df = pd.DataFrame(top_entities, columns=["entity", "count"])
ent_df["text"] = ent_df["entity"].apply(lambda x: x[0])
ent_df["label"] = ent_df["entity"].apply(lambda x: x[1])

fig = px.bar(
    ent_df,
    x="count",
    y="text",
    color="label",
    orientation="h",
    title="🌍 Top Named Entities"
)
fig.show()

## 6. Países mais citados

In [20]:
countries = []

for ents in df["entities"].dropna():
    for text, label in ents:
        if label in ["GPE", "LOC"]:
            countries.append(text)

country_counts = Counter(countries).most_common(15)

country_df = pd.DataFrame(country_counts, columns=["country", "count"])

fig = px.bar(
    country_df,
    x="count",
    y="country",
    orientation="h",
    title="🗺️ Most Mentioned Locations"
)
fig.show()

## 8. Busca semântica (usando embeddings)

In [24]:
from sklearn.metrics.pairwise import cosine_similarity

def search(query, df, top_k=5):
    from nlp_pipeline import get_embedding
    
    q_emb = np.array(get_embedding(query)).reshape(1, -1)
    
    doc_embs = np.array([
        e for e in df["embedding"] if e
    ])
    
    sims = cosine_similarity(q_emb, doc_embs)[0]
    
    top_idx = sims.argsort()[-top_k:][::-1]
    
    results = df.iloc[top_idx][["title", "link"]]
    
    return results

# exemplo
search("AI regulation China US", df)

,title,link
127,A tech worker in China is laid off and replace...,https://www.npr.org/2026/05/01/nx-s1-5807131/t...
41,Why is China banning drone sales in Beijing?Th...,https://www.bbc.com/news/videos/czd21r472nmo
12,China scraps tariffs for all but one African n...,https://www.bbc.com/news/articles/cwy2v509217o
55,2 days agoMeta shares slide as plan to spend b...,https://www.bbc.com/news/articles/crkpd4r2y7eo
91,More private health records of UK Biobank volu...,https://www.theguardian.com/technology/2026/ap...


## Distribuição por fonte

In [23]:
fig = px.pie(
    df,
    names="source",
    title="📰 Source Distribution",
    hole=0.3
)
fig.show()